# AnimalCLEF 2026 — DINOv2 Re-ID Pipeline
Fine-tunes ViT-Base/16 (DINOv2) with ArcFace loss on known identities,
then uses MLS thresholding for open-set detection and DBSCAN for unknown individual clustering.

In [ ]:
# ── Install deps (only needed in Colab) ──────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q timm wildlife-datasets wildlife-tools

In [ ]:
# ── Mount Drive in Colab, or use local root ───────────────────────────────────
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust to wherever you uploaded the competition data on Drive
    ROOT = '/content/drive/MyDrive/animalCLEF-2026/'
else:
    ROOT = '/Users/angelazhu/code/courses/ai535_deeplearning/animalCLEF_project/animalCLEF-2026/'

print('ROOT:', ROOT)
assert os.path.exists(ROOT), f'ROOT not found: {ROOT}'

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import torch

CFG = dict(
    model_name        = 'vit_base_patch16_224.dino',
    img_size          = 224,
    embed_dim         = 768,      # ViT-Base output dim; set smaller (e.g. 512) to project down
    arcface_s         = 30.0,     # scale
    arcface_m         = 0.50,     # margin
    batch_size        = 32,
    num_epochs        = 10,
    lr_head           = 1e-3,     # head + projector LR during warmup
    lr_backbone       = 1e-5,     # last 4 ViT blocks LR during fine-tune
    warmup_epochs     = 2,        # freeze backbone for this many epochs first
    mls_threshold     = None,     # auto-calibrated from train MLS distribution
    mls_pct           = 5,        # percentile of train MLS used as threshold
    dbscan_eps        = 0.5,      # cosine distance cutoff for DBSCAN
    dbscan_min_samples= 2,
    device            = 'cuda' if torch.cuda.is_available() else 'cpu',
    seed              = 42,
)

print('Device:', CFG['device'])

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

## 1. Data Loading

In [ ]:
# ── Load metadata ─────────────────────────────────────────────────────────────
meta = pd.read_csv(os.path.join(ROOT, 'metadata.csv'))
print(meta.groupby(['dataset', 'split']).size().to_string())
print(f"\nTotal images: {len(meta)}")
print(f"Unknown (no identity label): {meta['identity'].isna().sum()}")

In [ ]:
# ── Build train / query splits ────────────────────────────────────────────────
# Known species have train images with identity labels.
# TexasHornedLizards has NO train data -> the open-set 'unknown' species.

train_df = meta[(meta['split'] == 'train') & (meta['identity'].notna())].copy()
query_df = meta[meta['split'] == 'test'].copy()

identity_classes = sorted(train_df['identity'].unique())
class_to_idx = {c: i for i, c in enumerate(identity_classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}
train_df['label'] = train_df['identity'].map(class_to_idx).astype(int)

NUM_CLASSES = len(identity_classes)
print(f'Known identities (train classes): {NUM_CLASSES}')
print(f'Train images : {len(train_df)}')
print(f'Query images : {len(query_df)}')
print(f'  known gt   : {query_df["identity"].notna().sum()}')
print(f'  unknown    : {query_df["identity"].isna().sum()}')

In [ ]:
# ── Dataset & loaders ─────────────────────────────────────────────────────────
class AnimalDataset(Dataset):
    def __init__(self, df, root, transform, label_col='label'):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.root, row['path'])).convert('RGB')
        img = self.transform(img)
        label = int(row[self.label_col]) if (self.label_col in row.index and pd.notna(row.get(self.label_col, None))) else -1
        return img, label


size = CFG['img_size']
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((size, size)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])
val_tf = T.Compose([
    T.Resize((size, size)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_ds = AnimalDataset(train_df, ROOT, train_tf, label_col='label')
query_ds = AnimalDataset(query_df, ROOT, val_tf,   label_col='label')

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
query_loader = DataLoader(query_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Query batches: {len(query_loader)}')

## 2. Model — DINOv2 ViT-Base + ArcFace Head

In [ ]:
# ── ArcFace loss ──────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss.
    Adds a fixed margin to the target class angle before softmax,
    pushing intra-class embeddings together and inter-class apart.
    """
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))  # (B, C)
        sine   = (1.0 - cosine.pow(2)).clamp(0, 1).sqrt()
        phi    = cosine * self.cos_m - sine * self.sin_m  # cos(theta + margin)
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1.0)
        logits  = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return F.cross_entropy(logits, labels), logits

In [ ]:
# ── DINOv2 Re-ID model ────────────────────────────────────────────────────────
class DINOv2ReID(nn.Module):
    def __init__(self, model_name, embed_dim, num_classes, s, m):
        super().__init__()
        # num_classes=0 → backbone returns raw [CLS] token embedding
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        bb_dim = self.backbone.num_features  # 768 for ViT-Base
        # Optional linear projection; keeps dim if embed_dim == bb_dim
        self.projector = (
            nn.Sequential(nn.Linear(bb_dim, embed_dim), nn.BatchNorm1d(embed_dim))
            if embed_dim != bb_dim else nn.Identity()
        )
        self.arcface = ArcFaceLoss(embed_dim, num_classes, s=s, m=m)

    def embed(self, x):
        """Return L2-normalised embedding (used at inference)."""
        return F.normalize(self.projector(self.backbone(x)), dim=1)

    def forward(self, x, labels=None):
        emb = self.embed(x)
        if labels is not None:
            loss, logits = self.arcface(emb, labels)
            return loss, logits, emb
        return emb

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_blocks(self, n=4):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in list(self.backbone.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True


model = DINOv2ReID(
    model_name=CFG['model_name'],
    embed_dim=CFG['embed_dim'],
    num_classes=NUM_CLASSES,
    s=CFG['arcface_s'],
    m=CFG['arcface_m'],
).to(CFG['device'])

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total params: {total_params:.1f}M')

## 3. Training

In [ ]:
# ── Optimiser factory ─────────────────────────────────────────────────────────
def make_optimizer(model, phase):
    if phase == 'warmup':
        model.freeze_backbone()
        params = [
            {'params': model.projector.parameters(), 'lr': CFG['lr_head']},
            {'params': model.arcface.parameters(),   'lr': CFG['lr_head']},
        ]
    else:  # finetune
        model.unfreeze_last_blocks(n=4)
        params = [
            {'params': [p for p in model.backbone.parameters() if p.requires_grad],
             'lr': CFG['lr_backbone']},
            {'params': model.projector.parameters(), 'lr': CFG['lr_head']},
            {'params': model.arcface.parameters(),   'lr': CFG['lr_head']},
        ]
    return torch.optim.AdamW(params, weight_decay=1e-4)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss, _, emb = model(imgs, labels)
        loss.backward()
        optimizer.step()

        # Accuracy on raw cosine logits (no margin) — ArcFace logits are
        # intentionally hard so argmax during training is meaningless
        with torch.no_grad():
            cosine_logits = F.linear(F.normalize(emb.detach()),
                                     F.normalize(model.arcface.weight))
        total_loss += loss.item() * imgs.size(0)
        correct    += (cosine_logits.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


history   = []
optimizer = make_optimizer(model, 'warmup')

for epoch in range(1, CFG['num_epochs'] + 1):
    if epoch == CFG['warmup_epochs'] + 1:
        print('--- Switching to backbone fine-tuning (last 4 blocks) ---')
        optimizer = make_optimizer(model, 'finetune')

    loss, acc = train_one_epoch(model, train_loader, optimizer, CFG['device'])
    history.append({'epoch': epoch, 'loss': loss, 'acc': acc})
    print(f'Epoch {epoch:02d}/{CFG["num_epochs"]}  loss={loss:.4f}  train_acc={acc:.4f}')

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hist_df['epoch'], hist_df['loss']);  axes[0].set(title='Loss',          xlabel='Epoch')
axes[1].plot(hist_df['epoch'], hist_df['acc']);   axes[1].set(title='Train Accuracy', xlabel='Epoch')
plt.tight_layout(); plt.show()

## 4. Feature Extraction

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device):
    """Returns (embeddings, raw_cosine_logits, labels) as numpy arrays."""
    model.eval()
    embs, logits_all, labels_all = [], [], []
    for imgs, labels in loader:
        imgs  = imgs.to(device)
        emb   = model.embed(imgs)                                           # (B, D)
        # Raw cosine logits — no margin, used for MLS scoring
        logit = F.linear(emb, F.normalize(model.arcface.weight))            # (B, C)
        embs.append(emb.cpu())
        logits_all.append(logit.cpu())
        labels_all.append(labels)
    return (
        torch.cat(embs).numpy(),
        torch.cat(logits_all).numpy(),
        torch.cat(labels_all).numpy(),
    )


print('Extracting train embeddings...')
train_embs, train_logits, train_labels = extract_features(model, train_loader, CFG['device'])

print('Extracting query embeddings...')
query_embs, query_logits, query_labels = extract_features(model, query_loader, CFG['device'])

print(f'train_embs : {train_embs.shape}')
print(f'query_embs : {query_embs.shape}')

## 5. Open-Set Detection via MLS

In [ ]:
# ── Maximum Logit Score (MLS) ─────────────────────────────────────────────────
# High MLS = the model is confident this is a known individual.
# Low MLS  = likely an unknown individual from an unseen species.

train_mls = train_logits.max(axis=1)  # (N_train,)
query_mls = query_logits.max(axis=1)  # (N_query,)

# Calibrate from the training distribution.
# Lower percentile = more permissive (fewer unknowns flagged).
if CFG['mls_threshold'] is None:
    CFG['mls_threshold'] = float(np.percentile(train_mls, CFG['mls_pct']))
    print(f'Auto MLS threshold ({CFG["mls_pct"]}th pct of train MLS): {CFG["mls_threshold"]:.4f}')

plt.figure(figsize=(8, 4))
plt.hist(train_mls, bins=80, alpha=0.6, label='Train (known)')
plt.hist(query_mls, bins=80, alpha=0.6, label='Query (all)')
plt.axvline(CFG['mls_threshold'], color='red', linestyle='--',
            label=f'Threshold = {CFG["mls_threshold"]:.3f}')
plt.xlabel('Maximum Logit Score'); plt.ylabel('Count')
plt.legend(); plt.title('MLS Distribution — adjust mls_pct or mls_threshold in CFG')
plt.tight_layout(); plt.show()

In [ ]:
# ── Assign known predictions ──────────────────────────────────────────────────
is_known = query_mls >= CFG['mls_threshold']
print(f'Classified as known  : {is_known.sum()} / {len(is_known)}')
print(f'Classified as unknown: {(~is_known).sum()} / {len(is_known)}')

predicted_ids = np.full(len(query_df), 'unknown', dtype=object)

if is_known.any():
    pred_cls = query_logits[is_known].argmax(axis=1)
    predicted_ids[is_known] = [idx_to_class[c] for c in pred_cls]

## 6. Unknown Individual Clustering via DBSCAN

In [ ]:
# ── DBSCAN in cosine-distance space ──────────────────────────────────────────
# Embeddings are L2-normalised so cosine_distance = 1 - cosine_similarity.
# DBSCAN with precomputed distances groups unknowns into pseudo-individuals.

unknown_mask = ~is_known
unknown_embs = query_embs[unknown_mask]

if unknown_embs.shape[0] > 0:
    dist_matrix = pairwise_distances(unknown_embs, metric='cosine')
    db = DBSCAN(
        eps=CFG['dbscan_eps'],
        min_samples=CFG['dbscan_min_samples'],
        metric='precomputed',
    ).fit(dist_matrix)

    cluster_labels = db.labels_
    n_clusters = len(set(cluster_labels) - {-1})
    n_noise    = (cluster_labels == -1).sum()
    print(f'DBSCAN clusters: {n_clusters}  |  noise singletons: {n_noise}')

    unknown_indices = np.where(unknown_mask)[0]
    noise_counter   = n_clusters
    for idx, cl in zip(unknown_indices, cluster_labels):
        if cl == -1:
            predicted_ids[idx] = f'unknown_singleton_{noise_counter}'
            noise_counter += 1
        else:
            predicted_ids[idx] = f'unknown_cluster_{cl}'
else:
    print('No unknown samples detected — lower mls_threshold if unexpected.')

## 7. Submission

In [ ]:
submission = query_df[['image_id']].copy()
submission['identity'] = predicted_ids

print(submission['identity'].value_counts().head(20))
print(f'\nTotal rows: {len(submission)}')

out_path = os.path.join(ROOT, 'submission_dinov2.csv')
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

In [ ]:
ckpt_path = os.path.join(ROOT, 'dinov2_reid.pt')
torch.save({
    'model_state': model.state_dict(),
    'class_to_idx': class_to_idx,
    'cfg': CFG,
}, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')